# Sea Ice Diagnostics and LENS comparison

This notebook contains:
- Statistics of Ice Area, Ice Volume, and Snow Volume

In [ ]:
import os

import xarray as xr
import statistics
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import nc_time_axis

import cupid_utils as cu

In [ ]:
CESM_output_dir = "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"
ts_dir = None  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"
base_case_output_dir = None  # None => use CESM_output_dir
case_name = "b.e30_alpha09d_m.B1850C_MTso_Gris_Marbl.ne30_t233_wgx3.377"  # "b.e30_beta02.BLT1850.ne30_t232.104"
case_nickname = "BLT1850_377"  # "BLT1850_104"
base_case_name = "b.e30_alpha09d_m.B1850C_MTso_Gris_Marbl.ne30_t233_wgx3.376"  # "b.e23_alpha17f.BLT1850.ne30_t233.092"
base_case_nickname = "BLT1850_376"  # "BLT1850_092"

start_date = "0001-01-01"  # "0001-01-01"
end_date = "0144-01-01"  # "0100-01-01"
align = 0
base_start_date = "0001-01-01"  # "0001-01-01"
base_end_date = "0103-01-01"  # "0100-01-01"
base_align = 0

obs_data_dir = "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_obs_data"  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_obs_data"
path_model = "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_model_data/ice/"  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_model_data/ice/"
grid_file = "/glade/campaign/cesm/community/omwg/grids/tx2_3v3_grid.nc"  # "/glade/campaign/cesm/community/omwg/grids/tx2_3v2_grid.nc"
climo_nyears = 35

serial = False  # use dask LocalCluster

lc_kwargs = {}

In [ ]:
# Want some base case parameter defaults to equal control case values
if base_case_name is not None:
    if base_case_output_dir is None:
        base_case_output_dir = CESM_output_dir

    if base_end_date is None:
        base_end_date = end_date

if ts_dir is None:
    ts_dir = CESM_output_dir

In [ ]:
# When running interactively, cupid_run should be set to 0 for
# a DASK cluster

cupid_run = 1

if cupid_run == 1:

    from dask.distributed import Client, LocalCluster

    # Spin up cluster (if running in parallel)
    client = None
    if not serial:
        cluster = LocalCluster(**lc_kwargs)
        client = Client(cluster)

else:

    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    cluster = PBSCluster(
        cores=16,
        processes=16,
        memory="100GB",
        account="P93300065",
        queue="casper",
        walltime="02:00:00",
    )

    client = Client(cluster)

    cluster.scale(1)

    print(cluster)

client

In [ ]:
# Read in three cases. The ADF timeseries are needed here.

ds1 = xr.open_mfdataset(
    os.path.join(
        ts_dir, case_name, "ice", "proc", "tseries", f"{case_name}.cice.h.*.nc"
    ),
    data_vars="minimal",
    compat="override",
    coords="minimal",
).sel(time=slice(start_date, end_date))

ds2 = xr.open_mfdataset(
    os.path.join(
        ts_dir,
        base_case_name,
        "ice",
        "proc",
        "tseries",
        f"{base_case_name}.cice.h.*.nc",
    ),
    data_vars="minimal",
    compat="override",
    coords="minimal",
).sel(time=slice(base_start_date, base_end_date))

ds_grid = xr.open_dataset(grid_file)


if "TLAT" in ds_grid:
    TLAT = ds_grid["TLAT"]
    TLON = ds_grid["TLONG"]
    tarea = ds_grid["TAREA"] * 1.0e-4
    angle = ds_grid["ANGLE"]
else:
    TLAT = ds_grid["tlat"] * 180.0 / np.pi
    TLON = ds_grid["tlon"] * 180.0 / np.pi
    htn = ds_grid["htn"] * 1.0e-2
    hte = ds_grid["hte"] * 1.0e-2
    tarea = htn * hte
    angle = ds_grid["angle"]

ds1_ann = ds1.resample(time="YS").mean(dim="time")
ds2_ann = ds2.resample(time="YS").mean(dim="time")

climo_nyears1 = min(climo_nyears, len(ds1_ann.time))
climo_nyears2 = min(climo_nyears, len(ds2_ann.time))

yml_dir = os.path.join(cu.__path__[0], "..", "input_files", "ice")
with open(os.path.join(yml_dir, "cice_masks.yml"), "r") as file:
    cice_masks = yaml.safe_load(file)

first_year = int(start_date.split("-")[0])
base_first_year = int(base_start_date.split("-")[0])
end_year = int(end_date.split("-")[0])
base_end_year = int(base_end_date.split("-")[0])

path_lens1 = os.path.join(path_model, "cesm_lens1")
path_lens2 = os.path.join(path_model, "cesm_lens2")

path_cdr = os.path.join(
    obs_data_dir, "ice", "analysis_datasets", "hemispheric_data", "CDR_area_timeseries/"
)

In [ ]:
### Read in the CESM LENS historical data

ds_cesm1_aicetot_nh = xr.open_dataset(path_lens1 + "/LE_aicetot_nh_1920-2100.nc")
ds_cesm1_hitot_nh = xr.open_dataset(path_lens1 + "/LE_hitot_nh_1920-2100.nc")
ds_cesm1_hstot_nh = xr.open_dataset(path_lens1 + "/LE_hstot_nh_1920-2100.nc")

ds_cesm1_aicetot_sh = xr.open_dataset(path_lens1 + "/LE_aicetot_sh_1920-2100.nc")
ds_cesm1_hitot_sh = xr.open_dataset(path_lens1 + "/LE_hitot_sh_1920-2100.nc")
ds_cesm1_hstot_sh = xr.open_dataset(path_lens1 + "/LE_hstot_sh_1920-2100.nc")

cesm1_aicetot_nh_ann = ds_cesm1_aicetot_nh["aice_monthly"].mean(dim="nmonth")
cesm1_hitot_nh_ann = ds_cesm1_hitot_nh["hi_monthly"].mean(dim="nmonth")
cesm1_hstot_nh_ann = ds_cesm1_hstot_nh["hs_monthly"].mean(dim="nmonth")

cesm1_aicetot_sh_ann = ds_cesm1_aicetot_sh["aice_monthly"].mean(dim="nmonth")
cesm1_hitot_sh_ann = ds_cesm1_hitot_sh["hi_monthly"].mean(dim="nmonth")
cesm1_hstot_sh_ann = ds_cesm1_hstot_sh["hs_monthly"].mean(dim="nmonth")

if first_year + align >= 1850 or base_first_year + base_align >= 1850:
    cesm1_years = np.linspace(1920, 2100, 181)
else:
    cesm1_years = np.linspace(1, 181, 181)

if first_year + align >= 1850 or base_first_year + base_align >= 1850:
    cesm1_climo_start = 60
    cesm1_climo_end = 95
else:
    cesm1_climo_start = 10
    cesm1_climo_end = 45

cesm1_aicetot_nh_month = ds_cesm1_aicetot_nh["aice_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hitot_nh_month = ds_cesm1_hitot_nh["hi_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hstot_nh_month = ds_cesm1_hstot_nh["hs_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_aicetot_sh_month = ds_cesm1_aicetot_sh["aice_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hitot_sh_month = ds_cesm1_hitot_sh["hi_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hstot_sh_month = ds_cesm1_hstot_sh["hs_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")

ds_cesm2_aicetot_nh = xr.open_dataset(path_lens2 + "/LE2_aicetot_nh_1870-2100.nc")
ds_cesm2_hitot_nh = xr.open_dataset(path_lens2 + "/LE2_hitot_nh_1870-2100.nc")
ds_cesm2_hstot_nh = xr.open_dataset(path_lens2 + "/LE2_hstot_nh_1870-2100.nc")

ds_cesm2_aicetot_sh = xr.open_dataset(path_lens2 + "/LE2_aicetot_sh_1870-2100.nc")
ds_cesm2_hitot_sh = xr.open_dataset(path_lens2 + "/LE2_hitot_sh_1870-2100.nc")
ds_cesm2_hstot_sh = xr.open_dataset(path_lens2 + "/LE2_hstot_sh_1870-2100.nc")

cesm2_aicetot_nh_ann = ds_cesm2_aicetot_nh["aice_monthly"].mean(dim="nmonth")
cesm2_hitot_nh_ann = ds_cesm2_hitot_nh["hi_monthly"].mean(dim="nmonth")
cesm2_hstot_nh_ann = ds_cesm2_hstot_nh["hs_monthly"].mean(dim="nmonth")

cesm2_aicetot_sh_ann = ds_cesm2_aicetot_sh["aice_monthly"].mean(dim="nmonth")
cesm2_hitot_sh_ann = ds_cesm2_hitot_sh["hi_monthly"].mean(dim="nmonth")
cesm2_hstot_sh_ann = ds_cesm2_hstot_sh["hs_monthly"].mean(dim="nmonth")

if first_year + align >= 1850 or base_first_year + base_align >= 1850:
    cesm2_years = np.linspace(1870, 2100, 231)
else:
    cesm2_years = np.linspace(1, 231, 231)

if first_year + align >= 1850 or base_first_year + base_align >= 1850:
    cesm2_climo_start = 110
    cesm2_climo_end = 145
else:
    cesm2_climo_start = 61
    cesm2_climo_end = 96

cesm2_aicetot_nh_month = ds_cesm2_aicetot_nh["aice_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hitot_nh_month = ds_cesm2_hitot_nh["hi_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hstot_nh_month = ds_cesm2_hstot_nh["hs_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_aicetot_sh_month = ds_cesm2_aicetot_sh["aice_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hitot_sh_month = ds_cesm2_hitot_sh["hi_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hstot_sh_month = ds_cesm2_hstot_sh["hs_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")

In [ ]:
cdr_nh = xr.open_dataset(path_cdr + "CDR_sic_nh_monthly.nc")
cdr_sh = xr.open_dataset(path_cdr + "CDR_sic_sh_monthly.nc")
cdr_lab = xr.open_dataset(path_cdr + "CDR_sic_lab_monthly.nc")

cdr_nh_mar = cdr_nh["sic_monthly"].sel(time=(cdr_nh.time.dt.month == 3))
cdr_sh_feb = cdr_sh["sic_monthly"].sel(time=(cdr_sh.time.dt.month == 2))
cdr_nh_sep = cdr_nh["sic_monthly"].sel(time=(cdr_nh.time.dt.month == 9))
cdr_sh_sep = cdr_sh["sic_monthly"].sel(time=(cdr_sh.time.dt.month == 9))
cdr_lab_mar = cdr_lab["sic_monthly_lab"].sel(time=(cdr_lab.time.dt.month == 3))

cdr_nh_clim = (
    cdr_nh["sic_monthly"]
    .isel(time=slice(-420, None))
    .groupby("time.month")
    .mean(dim="time", skipna=True)
)

cdr_sh_clim = (
    cdr_sh["sic_monthly"]
    .isel(time=slice(-420, None))
    .groupby("time.month")
    .mean(dim="time", skipna=True)
)

In [ ]:
cesm1_aicetot_nh_mar = ds_cesm1_aicetot_nh["aice_monthly"].isel(nmonth=3)
cesm1_hitot_nh_mar = ds_cesm1_hitot_nh["hi_monthly"].isel(nmonth=3)
cesm1_hstot_nh_mar = ds_cesm1_hstot_nh["hs_monthly"].isel(nmonth=3)

cesm1_aicetot_nh_sep = ds_cesm1_aicetot_nh["aice_monthly"].isel(nmonth=9)
cesm1_hitot_nh_sep = ds_cesm1_hitot_nh["hi_monthly"].isel(nmonth=9)
cesm1_hstot_nh_sep = ds_cesm1_hstot_nh["hs_monthly"].isel(nmonth=9)

cesm1_aicetot_sh_feb = ds_cesm1_aicetot_sh["aice_monthly"].isel(nmonth=2)
cesm1_hitot_sh_feb = ds_cesm1_hitot_sh["hi_monthly"].isel(nmonth=2)
cesm1_hstot_sh_feb = ds_cesm1_hstot_sh["hs_monthly"].isel(nmonth=2)

cesm1_aicetot_sh_sep = ds_cesm1_aicetot_sh["aice_monthly"].isel(nmonth=9)
cesm1_hitot_sh_sep = ds_cesm1_hitot_sh["hi_monthly"].isel(nmonth=9)
cesm1_hstot_sh_sep = ds_cesm1_hstot_sh["hs_monthly"].isel(nmonth=9)

In [ ]:
cesm2_aicetot_nh_mar = ds_cesm2_aicetot_nh["aice_monthly"].isel(nmonth=3)
cesm2_hitot_nh_mar = ds_cesm2_hitot_nh["hi_monthly"].isel(nmonth=3)
cesm2_hstot_nh_mar = ds_cesm2_hstot_nh["hs_monthly"].isel(nmonth=3)

cesm2_aicetot_nh_sep = ds_cesm2_aicetot_nh["aice_monthly"].isel(nmonth=9)
cesm2_hitot_nh_sep = ds_cesm2_hitot_nh["hi_monthly"].isel(nmonth=9)
cesm2_hstot_nh_sep = ds_cesm2_hstot_nh["hs_monthly"].isel(nmonth=9)

cesm2_aicetot_sh_feb = ds_cesm2_aicetot_sh["aice_monthly"].isel(nmonth=2)
cesm2_hitot_sh_feb = ds_cesm2_hitot_sh["hi_monthly"].isel(nmonth=2)
cesm2_hstot_sh_feb = ds_cesm2_hstot_sh["hs_monthly"].isel(nmonth=2)

cesm2_aicetot_sh_sep = ds_cesm2_aicetot_sh["aice_monthly"].isel(nmonth=9)
cesm2_hitot_sh_sep = ds_cesm2_hitot_sh["hi_monthly"].isel(nmonth=9)
cesm2_hstot_sh_sep = ds_cesm2_hstot_sh["hs_monthly"].isel(nmonth=9)

In [ ]:
ds1_area_nh = (tarea * ds1.aice).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-12
ds2_area_nh = (tarea * ds2.aice).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-12

ds1_icevol_nh = (tarea * ds1.hi).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-13
ds2_icevol_nh = (tarea * ds2.hi).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-13

ds1_snovol_nh = (tarea * ds1.hs).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-13
ds2_snovol_nh = (tarea * ds2.hs).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT > 0
).sum(dim=["nj", "ni"]) * 1.0e-13

ds1_area_sh = (tarea * ds1.aice).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-12
ds2_area_sh = (tarea * ds2.aice).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-12

ds1_icevol_sh = (tarea * ds1.hi).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-13
ds2_icevol_sh = (tarea * ds2.hi).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-13

ds1_snovol_sh = (tarea * ds1.hs).isel(time=slice(-climo_nyears1 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-13
ds2_snovol_sh = (tarea * ds2.hs).isel(time=slice(-climo_nyears2 * 12, None)).where(
    TLAT < 0
).sum(dim=["nj", "ni"]) * 1.0e-13

In [ ]:
### Northern Hemisphere

ds1_area_mar_nh = ds1_area_nh.sel(time=(ds1_area_nh.time.dt.month == 3)).load()
ds2_area_mar_nh = ds2_area_nh.sel(time=(ds2_area_nh.time.dt.month == 3)).load()

ds1_icevol_mar_nh = ds1_icevol_nh.sel(time=(ds1_icevol_nh.time.dt.month == 3)).load()
ds2_icevol_mar_nh = ds2_icevol_nh.sel(time=(ds2_icevol_nh.time.dt.month == 3)).load()

ds1_snovol_mar_nh = ds1_snovol_nh.sel(time=(ds1_snovol_nh.time.dt.month == 3)).load()
ds2_snovol_mar_nh = ds2_snovol_nh.sel(time=(ds2_snovol_nh.time.dt.month == 3)).load()

ds1_area_sep_nh = ds1_area_nh.sel(time=(ds1_area_nh.time.dt.month == 9)).load()
ds2_area_sep_nh = ds2_area_nh.sel(time=(ds2_area_nh.time.dt.month == 9)).load()

ds1_icevol_sep_nh = ds1_icevol_nh.sel(time=(ds1_icevol_nh.time.dt.month == 9)).load()
ds2_icevol_sep_nh = ds2_icevol_nh.sel(time=(ds2_icevol_nh.time.dt.month == 9)).load()

ds1_snovol_sep_nh = ds1_snovol_nh.sel(time=(ds1_snovol_nh.time.dt.month == 9)).load()
ds2_snovol_sep_nh = ds2_snovol_nh.sel(time=(ds2_snovol_nh.time.dt.month == 9)).load()

### Southern Hemisphere
ds1_area_feb_sh = ds1_area_sh.sel(time=(ds1_area_sh.time.dt.month == 2)).load()
ds2_area_feb_sh = ds2_area_sh.sel(time=(ds2_area_sh.time.dt.month == 2)).load()

ds1_icevol_feb_sh = ds1_icevol_sh.sel(time=(ds1_icevol_sh.time.dt.month == 2)).load()
ds2_icevol_feb_sh = ds2_icevol_sh.sel(time=(ds2_icevol_sh.time.dt.month == 2)).load()

ds1_snovol_feb_sh = ds1_snovol_sh.sel(time=(ds1_snovol_sh.time.dt.month == 2)).load()
ds2_snovol_feb_sh = ds2_snovol_sh.sel(time=(ds2_snovol_sh.time.dt.month == 2)).load()

ds1_area_sep_sh = ds1_area_sh.sel(time=(ds1_area_sh.time.dt.month == 9)).load()
ds2_area_sep_sh = ds2_area_sh.sel(time=(ds2_area_sh.time.dt.month == 9)).load()

ds1_icevol_sep_sh = ds1_icevol_sh.sel(time=(ds1_icevol_sh.time.dt.month == 9)).load()
ds2_icevol_sep_sh = ds2_icevol_sh.sel(time=(ds2_icevol_sh.time.dt.month == 9)).load()

ds1_snovol_sep_sh = ds1_snovol_sh.sel(time=(ds1_snovol_sh.time.dt.month == 9)).load()
ds2_snovol_sep_sh = ds2_snovol_sh.sel(time=(ds2_snovol_sh.time.dt.month == 9)).load()

In [ ]:
cesm1_aicetot_nh_mar_mean = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_max = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_min = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_std = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_nh_mar_mean = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_max = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_min = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_std = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_nh_mar_mean = cdr_nh_mar.mean()
cdr_nh_mar_max = cdr_nh_mar.max()
cdr_nh_mar_min = cdr_nh_mar.min()
cdr_nh_mar_std = cdr_nh_mar.std()

ds1_area_mar_nh_mean = ds1_area_mar_nh[-climo_nyears1::].mean(dim="time")
ds2_area_mar_nh_mean = ds2_area_mar_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_aicetot_nh_sep_mean = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_max = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_min = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_std = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_nh_sep_mean = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_max = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_min = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_std = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_nh_sep_mean = cdr_nh_sep.mean()
cdr_nh_sep_max = cdr_nh_sep.max()
cdr_nh_sep_min = cdr_nh_sep.min()
cdr_nh_sep_std = cdr_nh_sep.std()

ds1_area_sep_nh_mean = ds1_area_sep_nh[-climo_nyears1::].mean(dim="time")
ds2_area_sep_nh_mean = ds2_area_sep_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hitot_nh_mar_mean = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_max = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_min = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_std = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_nh_mar_mean = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_max = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_min = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_std = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_icevol_mar_nh_mean = ds1_icevol_mar_nh[-climo_nyears1::].mean(dim="time")
ds2_icevol_mar_nh_mean = ds2_icevol_mar_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hitot_nh_sep_mean = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_max = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_min = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_std = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_nh_sep_mean = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_max = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_min = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_std = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_icevol_sep_nh_mean = ds1_icevol_sep_nh[-climo_nyears1::].mean(dim="time")
ds2_icevol_sep_nh_mean = ds2_icevol_sep_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hstot_nh_mar_mean = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_max = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_min = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_std = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_nh_mar_mean = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_max = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_min = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_std = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_snovol_mar_nh_mean = ds1_snovol_mar_nh[-climo_nyears1::].mean(dim="time")
ds2_snovol_mar_nh_mean = ds2_snovol_mar_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hstot_nh_sep_mean = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_max = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_min = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_std = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_nh_sep_mean = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_max = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_min = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_std = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_snovol_sep_nh_mean = ds1_snovol_sep_nh[-climo_nyears1::].mean(dim="time")
ds2_snovol_sep_nh_mean = ds2_snovol_sep_nh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_aicetot_sh_feb_mean = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_max = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_min = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_std = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_sh_feb_mean = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_max = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_min = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_std = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_sh_feb_mean = cdr_sh_feb.mean()
cdr_sh_feb_max = cdr_sh_feb.max()
cdr_sh_feb_min = cdr_sh_feb.min()
cdr_sh_feb_std = cdr_sh_feb.std()

ds1_area_feb_sh_mean = ds1_area_feb_sh[-climo_nyears1::].mean(dim="time")
ds2_area_feb_sh_mean = ds2_area_feb_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_aicetot_sh_sep_mean = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_max = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_min = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_std = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_sh_sep_mean = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_max = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_min = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_std = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_sh_sep_mean = cdr_sh_sep.mean()
cdr_sh_sep_max = cdr_sh_sep.max()
cdr_sh_sep_min = cdr_sh_sep.min()
cdr_sh_sep_std = cdr_sh_sep.std()

ds1_area_sep_sh_mean = ds1_area_sep_sh[-climo_nyears1::].mean(dim="time")
ds2_area_sep_sh_mean = ds2_area_sep_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hitot_sh_feb_mean = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_max = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_min = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_std = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_sh_feb_mean = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_max = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_min = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_std = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_icevol_feb_sh_mean = ds1_icevol_feb_sh[-climo_nyears1::].mean(dim="time")
ds2_icevol_feb_sh_mean = ds2_icevol_feb_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hitot_sh_sep_mean = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_max = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_min = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_std = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_sh_sep_mean = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_max = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_min = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_std = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_icevol_sep_sh_mean = ds1_icevol_sep_sh[-climo_nyears1::].mean(dim="time")
ds2_icevol_sep_sh_mean = ds2_icevol_sep_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hstot_sh_feb_mean = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_max = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_min = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_std = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_sh_feb_mean = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_max = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_min = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_std = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_snovol_feb_sh_mean = ds1_snovol_feb_sh[-climo_nyears1::].mean(dim="time")
ds2_snovol_feb_sh_mean = ds2_snovol_feb_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
cesm1_hstot_sh_sep_mean = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_max = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_min = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_std = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_sh_sep_mean = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_max = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_min = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_std = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

ds1_snovol_sep_sh_mean = ds1_snovol_sep_sh[-climo_nyears1::].mean(dim="time")
ds2_snovol_sep_sh_mean = ds2_snovol_sep_sh[-climo_nyears2::].mean(dim="time")

In [ ]:
import pandas as pd

data = [
    [
        (ds1_area_mar_nh_mean - cesm1_aicetot_nh_mar_mean).values,
        (
            (ds1_area_mar_nh_mean - cesm1_aicetot_nh_mar_mean)
            / cesm1_aicetot_nh_mar_std
        ).values,
        (ds2_area_mar_nh_mean - cesm1_aicetot_nh_mar_mean).values,
        (
            (ds2_area_mar_nh_mean - cesm1_aicetot_nh_mar_mean)
            / cesm1_aicetot_nh_mar_std
        ).values,
    ],
    [
        (ds1_area_sep_nh_mean - cesm1_aicetot_nh_sep_mean).values,
        (
            (ds1_area_sep_nh_mean - cesm1_aicetot_nh_sep_mean)
            / cesm1_aicetot_nh_sep_std
        ).values,
        (ds2_area_sep_nh_mean - cesm1_aicetot_nh_sep_mean).values,
        (
            (ds2_area_sep_nh_mean - cesm1_aicetot_nh_sep_mean)
            / cesm1_aicetot_nh_sep_std
        ).values,
    ],
    [
        (ds1_area_mar_nh_mean - cdr_nh_mar_mean).values,
        ((ds1_area_mar_nh_mean - cdr_nh_mar_mean) / cdr_nh_mar_std).values,
        (ds2_area_mar_nh_mean - cdr_nh_mar_mean).values,
        ((ds2_area_mar_nh_mean - cdr_nh_mar_mean) / cdr_nh_mar_std).values,
    ],
    [
        (ds1_area_sep_nh_mean - cdr_nh_sep_mean).values,
        ((ds1_area_sep_nh_mean - cdr_nh_sep_mean) / cdr_nh_sep_std).values,
        (ds2_area_sep_nh_mean - cdr_nh_sep_mean).values,
        ((ds2_area_sep_nh_mean - cdr_nh_sep_mean) / cdr_nh_sep_std).values,
    ],
    [
        (ds1_icevol_mar_nh_mean - cesm1_hitot_nh_mar_mean).values,
        (
            (ds1_icevol_mar_nh_mean - cesm1_hitot_nh_mar_mean) / cesm1_hitot_nh_mar_std
        ).values,
        (ds2_icevol_mar_nh_mean - cesm1_hitot_nh_mar_mean).values,
        (
            (ds2_icevol_mar_nh_mean - cesm1_hitot_nh_mar_mean) / cesm1_hitot_nh_mar_std
        ).values,
    ],
    [
        (ds1_icevol_sep_nh_mean - cesm1_hitot_nh_sep_mean).values,
        (
            (ds1_icevol_sep_nh_mean - cesm1_hitot_nh_sep_mean) / cesm1_hitot_nh_sep_std
        ).values,
        (ds2_icevol_sep_nh_mean - cesm1_hitot_nh_sep_mean).values,
        (
            (ds2_icevol_sep_nh_mean - cesm1_hitot_nh_sep_mean) / cesm1_hitot_nh_sep_std
        ).values,
    ],
    [
        (ds1_snovol_mar_nh_mean - cesm2_hstot_nh_mar_mean).values,
        (
            (ds1_snovol_mar_nh_mean - cesm2_hstot_nh_mar_mean) / cesm2_hstot_nh_mar_std
        ).values,
        (ds2_snovol_mar_nh_mean - cesm2_hstot_nh_mar_mean).values,
        (
            (ds2_snovol_mar_nh_mean - cesm2_hstot_nh_mar_mean) / cesm2_hstot_nh_mar_std
        ).values,
    ],
    [
        (ds1_snovol_sep_nh_mean - cesm2_hstot_nh_sep_mean).values,
        (
            (ds1_snovol_sep_nh_mean - cesm2_hstot_nh_sep_mean) / cesm2_hstot_nh_sep_std
        ).values,
        (ds2_snovol_sep_nh_mean - cesm2_hstot_nh_sep_mean).values,
        (
            (ds2_snovol_sep_nh_mean - cesm2_hstot_nh_sep_mean) / cesm2_hstot_nh_sep_std
        ).values,
    ],
]

df = pd.DataFrame(
    data,
    columns=[
        case_nickname + "(diff)",
        case_nickname + "(std)",
        base_case_nickname + "(diff)",
        base_case_nickname + "(std)",
    ],
    index=[
        "Area vs CESM1 Mar",
        "Area vs CESM1 Sep",
        "Area vs CDR Mar",
        "Area vs CDR Sep",
        "Ice Volume vs CESM1 Mar",
        "Ice Volume vs CESM1 Sep",
        "Snow Volume vs CESM2 Mar",
        "Snow Volume vs CESM2 Sep",
    ],
)

from IPython.core.display import HTML
from IPython.display import display


def color_big_std(val):
    color = "white" if abs(val) < 3.0 else "cyan"
    return f"background-color: {color}"


styler = (
    df.style.map(color_big_std)
    .set_caption("Northern Hemisphere")
    .format("{:5.2f}".format)
)

display(HTML(styler.to_html(index=True)))

# df.to_csv("seaice.csv")

In [ ]:
data = [
    [
        (ds1_area_feb_sh_mean - cesm1_aicetot_sh_feb_mean).values,
        (
            (ds1_area_feb_sh_mean - cesm1_aicetot_sh_feb_mean)
            / cesm1_aicetot_sh_feb_std
        ).values,
        (ds2_area_feb_sh_mean - cesm1_aicetot_sh_feb_mean).values,
        (
            (ds2_area_feb_sh_mean - cesm1_aicetot_sh_feb_mean)
            / cesm1_aicetot_sh_feb_std
        ).values,
    ],
    [
        (ds1_area_sep_sh_mean - cesm1_aicetot_sh_sep_mean).values,
        (
            (ds1_area_sep_sh_mean - cesm1_aicetot_sh_sep_mean)
            / cesm1_aicetot_sh_sep_std
        ).values,
        (ds2_area_sep_sh_mean - cesm1_aicetot_sh_sep_mean).values,
        (
            (ds2_area_sep_sh_mean - cesm1_aicetot_sh_sep_mean)
            / cesm1_aicetot_sh_sep_std
        ).values,
    ],
    [
        (ds1_area_feb_sh_mean - cdr_sh_feb_mean).values,
        ((ds1_area_feb_sh_mean - cdr_sh_feb_mean) / cdr_sh_feb_std).values,
        (ds2_area_feb_sh_mean - cdr_sh_feb_mean).values,
        ((ds2_area_feb_sh_mean - cdr_sh_feb_mean) / cdr_sh_feb_std).values,
    ],
    [
        (ds1_area_sep_sh_mean - cdr_sh_sep_mean).values,
        ((ds1_area_sep_sh_mean - cdr_sh_sep_mean) / cdr_sh_sep_std).values,
        (ds2_area_sep_sh_mean - cdr_sh_sep_mean).values,
        ((ds2_area_sep_sh_mean - cdr_sh_sep_mean) / cdr_sh_sep_std).values,
    ],
    [
        (ds1_icevol_feb_sh_mean - cesm1_hitot_sh_feb_mean).values,
        (
            (ds1_icevol_feb_sh_mean - cesm1_hitot_sh_feb_mean) / cesm1_hitot_sh_feb_std
        ).values,
        (ds2_icevol_feb_sh_mean - cesm1_hitot_sh_feb_mean).values,
        (
            (ds2_icevol_feb_sh_mean - cesm1_hitot_sh_feb_mean) / cesm1_hitot_sh_feb_std
        ).values,
    ],
    [
        (ds1_icevol_sep_sh_mean - cesm1_hitot_sh_sep_mean).values,
        (
            (ds1_icevol_sep_sh_mean - cesm1_hitot_sh_sep_mean) / cesm1_hitot_sh_sep_std
        ).values,
        (ds2_icevol_sep_sh_mean - cesm1_hitot_sh_sep_mean).values,
        (
            (ds2_icevol_sep_sh_mean - cesm1_hitot_sh_sep_mean) / cesm1_hitot_sh_sep_std
        ).values,
    ],
    [
        (ds1_snovol_feb_sh_mean - cesm2_hstot_sh_feb_mean).values,
        (
            (ds1_snovol_feb_sh_mean - cesm2_hstot_sh_feb_mean) / cesm2_hstot_sh_feb_std
        ).values,
        (ds2_snovol_feb_sh_mean - cesm2_hstot_sh_feb_mean).values,
        (
            (ds2_snovol_feb_sh_mean - cesm2_hstot_sh_feb_mean) / cesm2_hstot_sh_feb_std
        ).values,
    ],
    [
        (ds1_snovol_sep_sh_mean - cesm2_hstot_sh_sep_mean).values,
        (
            (ds1_snovol_sep_sh_mean - cesm2_hstot_sh_sep_mean) / cesm2_hstot_sh_sep_std
        ).values,
        (ds2_snovol_sep_sh_mean - cesm2_hstot_sh_sep_mean).values,
        (
            (ds2_snovol_sep_sh_mean - cesm2_hstot_sh_sep_mean) / cesm2_hstot_sh_sep_std
        ).values,
    ],
]

df = pd.DataFrame(
    data,
    columns=[
        case_nickname + "(diff)",
        case_nickname + "(std)",
        base_case_nickname + "(diff)",
        base_case_nickname + "(std)",
    ],
    index=[
        "Area vs CESM1 Feb",
        "Area vs CESM1 Sep",
        "Area vs CDR Feb",
        "Area vs CDR Sep",
        "Ice Volume vs CESM1 Feb",
        "Ice Volume vs CESM1 Sep",
        "Snow Volume vs CESM2 Feb",
        "Snow Volume vs CESM2 Sep",
    ],
)

from IPython.core.display import HTML
from IPython.display import display


def color_big_std(val):
    color = "white" if abs(val) < 3.0 else "cyan"
    return f"background-color: {color}"


styler = (
    df.style.map(color_big_std)
    .set_caption("Southern Hemisphere")
    .format("{:5.2f}".format)
)

display(HTML(styler.to_html(index=True)))

In [ ]:
if client and not cupid_run:
    client.shutdown()